In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
import os
import json
import shutil
from datetime import datetime
from tqdm.notebook import tqdm

In [ ]:
# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.utils import resample

In [3]:
# Deep learning imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

# Set Random Seed

In [4]:
# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# 1. Load Dataset

In [ ]:
print("📚 Bloom's Taxonomy Classification Pipeline")
print("=" * 50)
print("\n1. 📁 Loading Dataset...")
try:
    df = pd.read_csv('/kaggle/input/blooms-taxonomy-dataset/blooms_taxonomy_dataset.csv')
    print(f"✅ Dataset loaded successfully!")
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print("Creating sample dataset for demonstration...")

    # Create sample data if file not found
    sample_data = {
        'Questions': [
            'What is the capital of France?',
            'Analyze the impact of climate change on agriculture',
            'Create a business plan for a new startup',
            'Remember the basic steps of photosynthesis',
            'Evaluate the effectiveness of different teaching methods',
            'List the primary colors',
            'Compare and contrast democracy and autocracy',
            'Design a new mobile application',
            'What are the main components of DNA?',
            'Assess the validity of this research study'
        ],
        'Category': ['BT1', 'BT4', 'BT5', 'BT1', 'BT5', 'BT1', 'BT4', 'BT5', 'BT1', 'BT5']
    }
    df = pd.DataFrame(sample_data)
    print("✅ Sample dataset created!")

# 2. Text Preprocessing

In [ ]:
print("\n2. 🧹 Text Preprocessing...")
def preprocess_text(text):
    """
    Preprocessing function to clean text data
    """
    if pd.isna(text):
        return ""

    # Convert to string and lowercase
    text = str(text).lower()

    # Remove URLs/links
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'www\.(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)

    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\?\!\-]', '', text)

    # Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

# Apply preprocessing
df['Questions_clean'] = df['Questions'].apply(preprocess_text)
print("✅ Text preprocessing completed!")

## Stratified Train/Val/Test Split

In [ ]:
import copy
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

UNDERSAMPLE = False   # Set False untuk skenario Tabel 4 (without_*), True untuk Tabel 5 (with_*)

# 1) Split stratified 80/20: test_df dipegang & tidak disentuh sampai evaluasi akhir
trainval_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['Category']
)

# 2) Buat validation set (10% dari trainval) untuk early stopping & seleksi model terbaik
train_df, val_df = train_test_split(
    trainval_df, test_size=0.1, random_state=42, stratify=trainval_df['Category']
)

# 3) Undersampling HANYA pada train_df (val_df dan test_df tetap distribusi asli)
if UNDERSAMPLE:
    n_min = train_df['Category'].value_counts().min()
    train_df = (train_df.groupby('Category', group_keys=False)
                .apply(lambda g: resample(g, replace=False, n_samples=n_min, random_state=42))
                .sample(frac=1, random_state=42).reset_index(drop=True))
    print(f"n_min: {n_min}; train after undersampling: {len(train_df)} ({train_df['Category'].nunique()} x n_min)")

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# 3. Display initial label distribution

In [ ]:
print("\n3. 📊 Initial Label Distribution:")
label_counts = df['Category'].value_counts().sort_index()
print(label_counts)

# Visualize distribution
plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
label_counts.plot(kind='bar', color='skyblue')
plt.title('Initial Label Distribution')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45)

## Prepare classifier inputs

In [ ]:
X_text_train = train_df['Questions_clean'].to_numpy()
X_text_val = val_df['Questions_clean'].to_numpy()
X_text_test = test_df['Questions_clean'].to_numpy()
y_train = train_df['Category'].to_numpy()
y_val = val_df['Category'].to_numpy()
y_test = test_df['Category'].to_numpy()

print(f"Train: {len(X_text_train)}, Val: {len(X_text_val)}, Test: {len(X_text_test)}")

# 7. BERT Setup

In [ ]:
print("\n7. 🤖 Setting up BERT...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased').to(device)

In [ ]:
# Freeze BERT parameters to reduce training time
for param in bert_model.parameters():
    param.requires_grad = False

print("✅ BERT model loaded and frozen!")

# 8. Dataset Class

In [ ]:
class BloomDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_encoder = LabelEncoder()
        self.label_encoder.fit(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.LongTensor([self.label_encoder.transform([self.labels[idx]])[0]])
        }

train_dataset = BloomDataset(X_text_train, y_train, tokenizer)
val_dataset = BloomDataset(X_text_val, y_val, tokenizer)
test_dataset = BloomDataset(X_text_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"✅ Datasets created: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")
print(f"   Classes: {train_dataset.label_encoder.classes_}")

# 9. Modified Model Architecture with Unfrozen BERT

In [ ]:
import torch.nn.functional as F

class BloomClassifier(nn.Module):
    def __init__(self, bert_model, num_classes, hidden_dim=256, variant='cnn_lstm', unfreeze_layers=2):
        super(BloomClassifier, self).__init__()
        assert variant in ['cnn', 'lstm', 'cnn_lstm'], "Invalid model variant"
        self.variant = variant
        self.bert = bert_model
        self.bert_dim = 768
        self.unfreeze_bert_layers(unfreeze_layers)

        if variant in ['cnn', 'cnn_lstm']:
            self.conv1 = nn.Conv1d(self.bert_dim, 128, kernel_size=3, padding=1)
            self.conv2 = nn.Conv1d(128, 64, kernel_size=3, padding=1)
            self.pool = nn.AdaptiveMaxPool1d(1)
        if variant in ['lstm', 'cnn_lstm']:
            self.lstm = nn.LSTM(self.bert_dim, hidden_dim // 2, batch_first=True,
                                bidirectional=True, dropout=0.2)

        combined_dim = (64 if variant in ['cnn', 'cnn_lstm'] else 0)
        combined_dim += hidden_dim if variant in ['lstm', 'cnn_lstm'] else 0
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def unfreeze_bert_layers(self, num_layers=2):
        for param in self.bert.parameters():
            param.requires_grad = False
        total_layers = len(self.bert.encoder.layer)
        for layer in self.bert.encoder.layer[total_layers - num_layers:]:
            for param in layer.parameters():
                param.requires_grad = True
        if getattr(self.bert, 'pooler', None) is not None:
            for param in self.bert.pooler.parameters():
                param.requires_grad = True

    def get_parameter_groups(self):
        bert_params = [p for p in self.bert.parameters() if p.requires_grad]
        other_params = [p for n, p in self.named_parameters()
                        if p.requires_grad and not n.startswith('bert')]
        return [
            {'params': bert_params, 'lr': 1e-5, 'name': 'bert'},
            {'params': other_params, 'lr': 2e-4, 'name': 'classifier'}
        ]

    def forward(self, input_ids, attention_mask):
        sequence_output = self.bert(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        features = []
        if self.variant in ['cnn', 'cnn_lstm']:
            cnn_input = sequence_output.permute(0, 2, 1)
            cnn_out = F.relu(self.conv1(cnn_input))
            cnn_out = F.relu(self.conv2(cnn_out))
            features.append(self.pool(cnn_out).squeeze(-1))
        if self.variant in ['lstm', 'cnn_lstm']:
            _, (hidden, _) = self.lstm(sequence_output)
            features.append(hidden.permute(1, 0, 2).contiguous().view(hidden.size(1), -1))
        combined = torch.cat(features, dim=1)
        combined = combined / (combined.norm(p=2, dim=1, keepdim=True) + 1e-7)
        return self.classifier(combined)

num_classes = len(train_dataset.label_encoder.classes_)
variant = 'cnn'  # or 'lstm', 'cnn_lstm'
model = BloomClassifier(bert_model, num_classes=num_classes, variant=variant, unfreeze_layers=2)
model.to(device)
print(f"✅ Model created: {sum(p.numel() for p in model.parameters()):,} parameters")

# 10. Training Setup with Discriminative Fine-tuning

In [ ]:
print("\n10. 🏋️ Training Setup with Discriminative Fine-tuning...")

import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        logpt = F.log_softmax(inputs, dim=1)
        pt = torch.exp(logpt)
        logpt = logpt.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = pt.gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = -self.alpha * (1 - pt) ** self.gamma * logpt
        return loss.mean() if self.reduction == 'mean' else loss.sum()

criterion = FocalLoss(alpha=1.0, gamma=2.0)

def setup_discriminative_optimizer(model, bert_lr=1e-5, classifier_lr=2e-4):
    param_groups = model.get_parameter_groups()
    param_groups[0]['lr'] = bert_lr
    param_groups[1]['lr'] = classifier_lr
    return optim.AdamW(param_groups, weight_decay=0.01)

optimizer = setup_discriminative_optimizer(model, bert_lr=1e-5, classifier_lr=2e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-7
)

best_val_loss, patience, patience_counter = float('inf'), 5, 0
best_model_state = None
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

epochs = 50
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].squeeze().to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        predicted = outputs.argmax(dim=1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    train_accuracy = 100 * train_correct / train_total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].squeeze().to(device)
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * val_correct / val_total
    scheduler.step()

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)
    print(f'Epoch [{epoch + 1}/{epochs}] Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch + 1}")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)


# 11. Training Visualization

In [ ]:
print("\n11. 📈 Training Visualization...")

plt.figure(figsize=(15, 5))

# Loss plot
plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Training Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy plot
plt.subplot(1, 3, 2)
plt.plot(train_accuracies, label='Training Accuracy', marker='o')
plt.plot(val_accuracies, label='Validation Accuracy', marker='s')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

# 12. Final Evaluation

In [ ]:
print("\n12. 📊 Final Evaluation...")

import json
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)

model.eval()
all_predictions, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['label'].squeeze().to(device)
        outputs = model(input_ids, attention_mask)
        all_predictions.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(targets.cpu().numpy())

pred_labels = train_dataset.label_encoder.inverse_transform(all_predictions)
true_labels = train_dataset.label_encoder.inverse_transform(all_labels)
labels = sorted(train_df['Category'].unique())
label_names = labels
acc = accuracy_score(true_labels, pred_labels)
p_w, r_w, f1_w, _ = precision_recall_fscore_support(
    true_labels, pred_labels, average='weighted', zero_division=0
)
f1_macro = f1_score(true_labels, pred_labels, average='macro', zero_division=0)
rep = classification_report(
    true_labels, pred_labels, labels=labels, output_dict=True, zero_division=0
)
cm = confusion_matrix(true_labels, pred_labels, labels=labels)
adj = {
    f"{labels[i]}<->{labels[i + 1]}": int(cm[i, i + 1] + cm[i + 1, i])
    for i in range(len(labels) - 1)
}
embedding_name = globals().get('embedding_name', 'bert-base-uncased')
keyword_mode = globals().get('keyword_mode', 'none')
with open(f"results_{embedding_name}_{variant}_{keyword_mode}_undersample{UNDERSAMPLE}.json", "w") as f:
    json.dump({
        'accuracy': acc, 'precision_w': p_w, 'recall_w': r_w,
        'f1_weighted': f1_w, 'f1_macro': f1_macro,
        'per_class': rep, 'cm': cm.tolist(), 'adjacent_confusions': adj
    }, f, indent=2)

print(f"Accuracy: {acc:.4f} | Precision (weighted): {p_w:.4f} | Recall (weighted): {r_w:.4f}")
print(f"F1 weighted: {f1_w:.4f} | F1 macro: {f1_macro:.4f}")
print(classification_report(true_labels, pred_labels, labels=labels, zero_division=0))
plt.subplot(1, 3, 3)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
accuracy, precision, recall, f1 = acc, p_w, r_w, f1_w

# 13. Save Model

In [ ]:
print("\n13. 💾 Saving Model...")
model_save_path = 'bloom_classifier_model.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'label_encoder': train_dataset.label_encoder,
    'model_config': {
        'num_classes': num_classes,
        'hidden_dim': 256
    }
}, model_save_path)

print(f"✅ Model saved to {model_save_path}")

# Summary
print(f"\n🎉 Pipeline Completed Successfully!")
print(f"   Final Accuracy: {accuracy:.4f}")
print(f"   Model saved: {model_save_path}")
print(f"   Total epochs trained: {len(train_losses)}")
print("=" * 50)

# Prediction Results

In [ ]:
print("\nDetailed Confusion Matrix Analysis:")
print("=" * 50)

# Print the Confusion Matrix
print("\nConfusion Matrix (Actual vs Predicted):")
print(cm)

# Print true and predicted labels for the test set
print("\nActual vs Predicted Labels for Test Data:")
print("-" * 30)

# Create a list of tuples (Actual, Predicted) for detailed viewing
test_results = list(zip(true_labels, pred_labels))

# You can print a few examples
print("First 10 examples:")
for i in range(min(10, len(test_results))):
    print(f"Example {i+1}: Actual = {test_results[i][0]}, Predicted = {test_results[i][1]}")

print("\nCounts of Actual and Predicted BT Categories in Test Set:")

# Use Counter to show distribution
print("\nActual Label Counts in Test Set:")
print(Counter(true_labels).most_common())

print("\nPredicted Label Counts in Test Set:")
print(Counter(pred_labels).most_common())

print("-" * 50)

# 14. Generate Misclassified Data

In [ ]:
print("\n14. ❌ Generating Misclassified Data...")

# Create a DataFrame for the test set with actual and predicted labels
test_df = pd.DataFrame({
    'Questions': X_text_test,
    'Actual': true_labels,
    'Predict': pred_labels
})

# Filter for misclassified instances
misclassified_df = test_df[test_df['Actual'] != test_df['Predict']]

print(f"✅ Found {len(misclassified_df)} misclassified instances.")

# Save misclassified data to CSV
misclassified_csv_path = 'misclassified_data.csv'
misclassified_df.to_csv(misclassified_csv_path, index=False)

print(f"✅ Misclassified data saved to {misclassified_csv_path}")
print("\nFirst 5 misclassified examples:")
print(misclassified_df.head())

print("=" * 50)

# Export Results

In [ ]:
# Generate timestamp and output folder name
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
folder_name = f"{variant}_{timestamp}"
os.makedirs(folder_name, exist_ok=True)

# Export steps
export_steps = [
    "Saving training & validation plots",
    "Saving confusion matrix (separate file)",
    "Saving metrics & classification report",
    "Saving predictions",
    "Saving misclassified data",
    "Saving model & zipping folder"
]

for step in tqdm(export_steps, desc="📦 Exporting Outputs", leave=False):
    if step == export_steps[0]:
        # === 1. Save Training & Validation Plots ===
        plt.figure(figsize=(15, 5))
        plt.subplot(1, 3, 1)
        plt.plot(train_losses, label='Training Loss', marker='o')
        plt.plot(val_losses, label='Validation Loss', marker='s')
        plt.title('Training and Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)

        plt.subplot(1, 3, 2)
        plt.plot(train_accuracies, label='Training Accuracy', marker='o')
        plt.plot(val_accuracies, label='Validation Accuracy', marker='s')
        plt.title('Training and Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy (%)')
        plt.legend()
        plt.grid(True)

        plt.subplot(1, 3, 3)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=label_names, yticklabels=label_names)
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')

        plt.tight_layout()
        training_plot_path = os.path.join(folder_name, f"training_plot_{variant}_{timestamp}.png")
        plt.savefig(training_plot_path)
        plt.close()

    elif step == export_steps[1]:
        # === Save Confusion Matrix as Separate File ===
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=label_names, yticklabels=label_names)
        plt.title('Test Set Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        confusion_matrix_path = os.path.join(folder_name, f"test_confusion_matrix_{variant}_{timestamp}.png")
        plt.tight_layout()
        plt.savefig(confusion_matrix_path)
        plt.close()

    elif step == export_steps[2]:
        # === Save Metrics & Report ===
        metrics_path = os.path.join(folder_name, f"metrics_{variant}_{timestamp}.txt")
        with open(metrics_path, 'w') as f:
            f.write(f"Accuracy: {accuracy:.4f}\n")
            f.write(f"Precision: {precision:.4f}\n")
            f.write(f"Recall: {recall:.4f}\n")
            f.write(f"F1-Score: {f1:.4f}\n")

        report_path = os.path.join(folder_name, f"classification_report_{variant}_{timestamp}.txt")
        with open(report_path, 'w') as f:
            f.write(classification_report(true_labels, pred_labels))

    elif step == export_steps[3]:
        # === Save Predictions ===
        predictions_path = os.path.join(folder_name, f"predictions_{variant}_{timestamp}.txt")
        with open(predictions_path, 'w') as f:
            for i, (actual, predicted) in enumerate(zip(true_labels, pred_labels)):
                f.write(f"{i+1}. Actual: {actual} | Predicted: {predicted}\n")

    elif step == export_steps[4]:
        # === Save Misclassified Data ===
        misclassified_path = os.path.join(folder_name, f"misclassified_{variant}_{timestamp}.csv")
        misclassified_df.to_csv(misclassified_path, index=False)

    elif step == export_steps[5]:
        # === Save Model & Zip Folder ===
        model_path = os.path.join(folder_name, f"model_{variant}_{timestamp}.pth")
        torch.save({
            'model_state_dict': model.state_dict(),
            'label_encoder': train_dataset.label_encoder,
            'model_config': {
                'num_classes': num_classes,
                'hidden_dim': 256,
                'variant': variant
            }
        }, model_path)

        zip_path = f"{folder_name}.zip"
        shutil.make_archive(folder_name, 'zip', folder_name)

print(f"\n✅ All outputs saved in: {folder_name}")
print(f"🗜️ Zipped archive created: {zip_path}")